# Team Member Contribution Table

| Member | Contribution |
| ------ | ------------ |
| Petter (SM) | Integrated posenet into FastAPI server. Created new endpoints for storing videos and issuing keypoint extraction. The keypoints, and their x, y positions are stored in a csv file.  |
| Kim | Learned and experimented with TensorFlow and Keras. |
| Elliot | Learned and experimented with TensorFlow and Keras. |
| Agust | Extended the web application to allow video upload to FastAPI server and issuing of keypoint extraction. |


# Integration of posenet

We chose to implement the pose estimation service in our server application (see `ml/api` project) by using the posenet module from `https://github.com/rwightman/posenet-pytorch` as a third party library (see `ml/api/app/third_party/posenet_pytorch`) combined with a utility file named `keypoint_extraction.py` (see `ml/api/app/utils/keypoint_extraction.py`) that we developed. The file extracts body keypoints with corresponding x, y coordinates from each frame in a given video and saves them in a csv format. The utility file is inspired from `image_demo.py` and `webcam_demo.py` from the linked repo as it combines some of the core functionalities from each to provide the keypoint extraction service. 

A client can use the service by posting a video on the server, then issue a keypoint extraction on that video that results in a csv file. The following API endpoints were developed to satisfy this usage: 

* /posenet/videos/upload - post a given video on the server
* /posenet/videos/list_videos - retrieve a list of the currently stored videos on the server
* /posenet/extraction/extract_keypoints - execute keypoint extraction on a video by posting the name of an existing video on the server
* /posenet/extraction/list_keypoint_files - retrieve a list of the currently stored keypoint extraction files (csv files)

The reader is referred to `ml/api/app/routers/videos.py` and `ml/api/app/routers/keypoint_extraction.py` for implementation details regarding these endpoints.

Two classes were created to manage videos and keypoint extraction files in the local file system: 
* VideoManager (see `ml/api/app/file_managers/video_manager.py`)
* KeypointManager (see `ml/api/app/file_managers/keypoint_manager.py`)

Each class is initialized with a base directory where videos/csv files are saved/loaded. The methods in the classes facilitates operations such as storing videos, listing available video/csv files, and of course, extracting and saving keypoints for a given video file in base directories. Remark that KeypointManager class uses a function named `run_keypoint_extraction` from `keypoint_extraction.py` to provide the extraction service. VideoManager and KeypointManager are initialized as global instances in `ml/api/dependencies/dependencies.py` which are then injected into the endpoints such that these are used by the server.

## Example keypoint extraction output

```csv
frame,pose_id,pose_score,keypoint,keypoint_score,y,x
0,0,0.66868053009624,nose,0.9984616041183472,162.65416597065172,557.4724288890291
0,0,0.66868053009624,leftEye,0.9990068078041077,148.07003288938287,576.7797002552112
0,0,0.66868053009624,rightEye,0.9984918832778931,146.7286521510074,541.276671962174
0,0,0.66868053009624,leftEar,0.792364239692688,167.51126004938493,595.2445617876335
0,0,0.66868053009624,rightEar,0.91425621509552,166.85693841231495,517.3565769091101
0,0,0.66868053009624,leftShoulder,0.9541110992431641,273.0846043636924,631.1299211850954
0,0,0.66868053009624,rightShoulder,0.687865138053894,269.2841084798177,478.28866914747056
0,0,0.66868053009624,leftElbow,0.7546055912971497,397.30622140984786,660.8224998716456
0,0,0.66868053009624,rightElbow,0.7235847115516663,389.2336072419819,422.5447929858639
0,0,0.66868053009624,leftWrist,0.9439380168914795,483.0489637140642,665.5588406983759
0,0,0.66868053009624,rightWrist,0.9371312856674194,512.1363040857148,414.401093364937
0,0,0.66868053009624,leftHip,0.5015100836753845,516.4795430501301,606.578122877474
```

The data above is a subset of the rows from a csv file containing extracted keypoints from a person posing in a video. The columns in the file describes the following: 
* frame - the given video frame
* pose_id - IDs of different poses in the frame. For instance, if two people are posing, then there would be a two different IDs identifying them: 0 and 1
* pose_score -  overall score of a given pose
* keypoint - keypoint on body such as nose, left eye, right shoulder, etc
* keypoint_score - how sure the model is in its prediction of a persons keypoint
* y - y coordinate of keypoint in the given frame
* x - x coordinate of keypoint in the given frame 


# Extended web application

!["None"](img/posenet_tab.png)

We have extended the web application with a new tab/index, which allows us to process videos through a browser interface.

* **Upload Video (left-side):** upload .mp4/.mov files to the server from users system and reveal the currently stored video list with a button.
* **Process Video (right-side):** select a previously uploaded video, and call FastAPI to generate CSV/keypoint files. CSV files can also be listed with the corresponding button.

**The following web-app endpoints were added:**
* `/api/video` -> forwards upload to `/posenet/videos/upload`
* `/api/videos` -> forwards list request to `/posenet/videos/list_videos`
* `/api/video/process` -> forwards processing request to `/posenet/extraction/extract_keypoints?video_name=...`
* `/api/csv-files` -> forwards CSV list request to `/posenet/extraction/list_keypoint_files`

The frontend logic also includes response parsing and error handling so users get clear status output when an upload, request or list retrieval is successful/unsuccessful.
